In [1]:
!pip install sentence-transformers chromadb transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Foun

In [2]:
import pandas as pd

#Loading the csv
df = pd.read_csv("/kaggle/input/datasets/thedevastator/pubmed-article-summarization-dataset/train.csv")

#Drop any rows with abstract missing
df = df.dropna(subset = ['abstract'])
print(f"{df.count()}")
df.head(5)

article     117232
abstract    119924
dtype: int64


,article,abstract
0,a recent systematic analysis showed that in 20...,background : the present study was carried out...
1,it occurs in more than 50% of patients and may...,backgroundanemia in patients with cancer who a...
2,"tardive dystonia ( td ) , a rarer side effect ...",tardive dystonia ( td ) is a serious side effe...
3,"lepidoptera include agricultural pests that , ...",many lepidopteran insects are agricultural pes...
4,syncope is caused by transient diffuse cerebra...,we present an unusual case of recurrent cough ...


In [3]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
import chromadb

df_subset = df.head(1000).copy()
#cleaning the database
def simpleclean(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().replace('\n',' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df_subset['abstract_cleaned'] = df_subset['abstract'].apply(simpleclean)

#Load the model to convert the text into embeddings
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

#The model wants a list 
print(f"Encoding {len(df_subset)} abstracts...")
abstract_list = df_subset['abstract_cleaned'].tolist()
embeddings = model.encode(abstract_list,show_progress_bar = True)
titles = [{"title": str(t)[:100]} for t in df_subset['article']]

client = chromadb.Client()

try:
    client.delete_collection(name = 'medical_docs')
except:
    pass
collection = client.create_collection(name = 'medical_docs')
collection.add(
    documents = abstract_list,
    embeddings = embeddings.tolist(),
    metadatas = titles,
    ids = [str(i) for i in range(len(df_subset))]
)

print(f"Done.{collection.count()} rows are now searchable ")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 1000 abstracts...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Done.1000 rows are now searchable 


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

print("Loading Gemma...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

def ask_medical_bot(question):
    query_vector = model.encode([question]).tolist()
    results = collection.query(query_embeddings=query_vector, n_results=3)
    context = ""
    for i in range(len(results['documents'][0])):
        title = results['metadatas'][0][i]['title']
        text = results['documents'][0][i]
        context += f"\nSource:{title}\nContent:{text}\n"

    prompt = f"""
    You are a medical research assistant. Use the provided research abstracts to answer the question.
    If the answer isn't in the context, say you don't know. 
    Always cite the "Source" title in your answer.

    CONTEXT:
    {context}

    QUESTION:
    {question}

    ANSWER:
    """
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = llm_model.generate(**inputs, max_new_tokens=300, temperature=0.1)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("ANSWER:")[-1]



Loading Gemma...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
print("\n--- TEST RUN ---")
response = ask_medical_bot("Tardive dystonia?")
print(response)


--- TEST RUN ---

     Tardive dystonia is a serious side effect of antipsychotic medications, more prevalent among typical antipsychotics. It is potentially irreversible in affected patients. Studies indicate that newer atypical antipsychotics have a lower risk of tardive dystonia compared to their counterparts. As a result, many clinicians may have developed a false sense of security regarding the use of these medications. The source provides a case study of a 20-year-old male with hyperthymic temperament and borderline intellectual functioning who experienced severe tardive dystonia after low-dose short-term exposure to risperidone and then olanzapine. The authors aim to alert readers about being judicious and cautious when prescribing antipsychotics, especially those with no core psychotic features, hyperthymic temperament, or borderline intellectual functioning, suggesting they are more susceptible to developing adverse effects such as tardive dystonia and to monitor the onset of